<a href="https://colab.research.google.com/github/HarryTran2811/SDN_IoT_IDS/blob/feature%2FQuan/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch transformers timm

In [ ]:
!pip install -q mamba_ssm einops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.2/307.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 4.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for mamba_ssm
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (mamba_ssm)


In [ ]:
! pip install -q lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.4/891.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 15.4 MB/s eta 0:00:00


In [ ]:
# Import libraries
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets
from torchvision.models import vit_b_16, swin_b, swin_v2_s
# from transformers import AutoModelForImageClassification, AutoFeatureExtractor
from torch.utils.data import DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
import matplotlib.pyplot as plt
import timm
from pytorch_lightning.callbacks.early_stopping import EarlyStopping

In [ ]:
! pip install gdown

In [ ]:
import gdown
import zipfile
import os

# Define file ID and download URL
file_id = '1mi4q2fAP9aV5c0WY14TOq65Kt20KTUeD'
download_url = f'https://drive.google.com/uc?id={file_id}'
output_path = 'cicids_2017_resized.zip'

# Download the file
gdown.download(download_url, output_path, quiet=False)

# Extract the zip file
with zipfile.ZipFile(output_path, 'r') as zip_ref:
    zip_ref.extractall('extracted_files')  # Specify extraction folder

# Clean up by deleting the downloaded zip file if not needed
os.remove(output_path)

Downloading...
From (original): https://drive.google.com/uc?id=1mi4q2fAP9aV5c0WY14TOq65Kt20KTUeD
From (redirected): https://drive.google.com/uc?id=1mi4q2fAP9aV5c0WY14TOq65Kt20KTUeD&confirm=t&uuid=b81ca661-d002-4dc1-beaf-0ba7cc9c6d21
To: /content/cicids_2017_resized.zip
100%|██████████| 3.82G/3.82G [00:43<00:00, 87.6MB/s]


In [ ]:
#Load generate dataset
data_dir = '/content/extracted_files/cicids_2017_resized'

In [ ]:
# Step 1: Define Data Augmentation and Preprocessing
data_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Step 2: Filter Dataset to Include Only Classes with >1 Image
def filter_dataset(dataset_path):
    filtered_classes = []
    for class_name in os.listdir(dataset_path):
        class_path = os.path.join(dataset_path, class_name)
        if os.path.isdir(class_path):
            files = os.listdir(class_path)
            if len(files) > 1:
                filtered_classes.append(class_name)
    return filtered_classes

In [ ]:
# Step 3: Load and Split the Dataset
filtered_classes = filter_dataset(data_dir)
filtered_dataset = datasets.ImageFolder(data_dir, transform=data_transforms)
filtered_dataset.samples = [s for s in filtered_dataset.samples if s[1] in [filtered_dataset.class_to_idx[c] for c in filtered_classes]]

dataset_size = len(filtered_dataset)
train_size = int(0.8 * dataset_size)
val_size = dataset_size - train_size
train_dataset, val_dataset = random_split(filtered_dataset, [train_size, val_size])

# Create dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
class TransferLearningVITModel(pl.LightningModule):
    def __init__(self, model_name="vit", num_classes=10, learning_rate=0.001):
        super(TransferLearningVITModel, self).__init__()
        self.learning_rate = learning_rate
        self.num_classes = num_classes

        # Load a pre-trained model
        if model_name == "vit":
            self.model = vit_b_16(pretrained=True)
            self.model.heads.head = nn.Linear(self.model.heads.head.in_features, num_classes)
        elif model_name == "swin-v2":
            self.model = swin_v2_s(pretrained=True)
            self.model.head = nn.Linear(self.model.head.in_features, num_classes)
        else:
            raise ValueError("Model name should be 'vit' or 'swin'")

        # Define loss function
        self.criterion = nn.CrossEntropyLoss()
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)
        acc = torch.sum(preds == labels).float() / len(labels)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)
        acc = torch.sum(preds == labels).float() / len(labels)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=0.9)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

In [ ]:
# Step 5: Initialize Model, Trainer, and Checkpoints
num_classes = len(filtered_dataset.classes)
# model = TransferLearningModel(num_classes=num_classes, learning_rate=0.001)
model = TransferLearningVITModel(model_name="vit", num_classes=num_classes, learning_rate=0.001)

# Define checkpoint directory and last checkpoint path
checkpoint_dir = "/content/checkpoints"
last_checkpoint_path = os.path.join(checkpoint_dir, "last-vit.ckpt")

checkpoint_callback = ModelCheckpoint(
    monitor="val_acc",
    mode="max",
    save_top_k=1,
    dirpath=checkpoint_dir,
    filename="best-checkpoint-vit",
    save_last=True
)
# Early stopping callback
early_stopping_callback = EarlyStopping(
    monitor="val_acc",
    mode="max",
    patience=5,
    verbose=True
)
# Load checkpoint if it exists
if os.path.exists(last_checkpoint_path):
    model = TransferLearningVITModel.load_from_checkpoint(last_checkpoint_path)
trainer = pl.Trainer(
    max_epochs=20,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1 if torch.cuda.is_available() else None,
    enable_progress_bar=True,
    callbacks=[checkpoint_callback,early_stopping_callback],
)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:04<00:00, 85.0MB/s]
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.util

In [ ]:
# Step 6: Train the Model
trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | model     | VisionTransformer | 85.8 M | train
1 | criterion | CrossEntropyLoss  | 0      | train
--------------------------------------------------------
85.8 M    Trainable params
0         Non-trainable params
85.8 M    Total params
343.235   Total estimated model params size (MB)
153       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_acc improved by 0.001 >= min_delta = 0.0. New best score: 1.000


Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_acc improved by 0.000 >= min_delta = 0.0. New best score: 1.000


In [ ]:
# Step 7: Load Best Checkpoint
best_model = TransferLearningVITModel.load_from_checkpoint(checkpoint_callback.best_model_path, num_classes=num_classes)
print("Best model loaded from checkpoint.")

# Step 8: Visualize Training and Validation Metrics
metrics = trainer.logged_metrics
train_losses = [metrics['train_loss'].item() for metrics in trainer.progress_bar_dict.values()]
val_losses = [metrics['val_loss'].item() for metrics in trainer.progress_bar_dict.values()]

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.legend()
plt.show()